# Scaling Laws

> Suppose you have a $100K compute budget. How deep and wide should the model be? How much data should you use? This question is not answered by intuition—the answer lies in a set of power-law relationships.
>
> This section starts from the basic concept of power-law distributions, walks through three paradigm shifts in Scaling Laws, and finally teaches you how to estimate the compute and memory required for training.

Scaling Laws study the quantitative relationships between model size, data volume, and training compute. The core finding is an empirical rule: increasing model size or adding data both reduce loss, but the reduction follows a power law—each doubling brings diminishing returns. This rule determines the optimal ratio of model to data for a given compute budget.

From Kaplan in 2020 to Chinchilla in 2022, and then to the overtraining paradigm after 2023, the conclusion about the optimal ratio was overturned twice. This section uses concrete numbers to trace the mathematical reasoning behind both shifts.

## First Half: Scaling Laws

## 1. Power-Law Distributions

Before diving into Scaling Laws, we need to understand the concept of a "power law."

#### 1.1 Linear Decay vs Power-Law Decay

Suppose the amount you save doubles each month:

- **Linear decay**: Each doubling, the gain decreases by a fixed amount
  - Save 100 → gain 10
  - Save 200 → gain 20 (+10)
  - Save 400 → gain 30 (+10) ← each doubling earns the same extra amount

- **Power-law decay**: Each doubling, the gain decreases by a fixed proportion
  - Save 100 → gain 10
  - Save 200 → gain 18 (+80%)
  - Save 400 → gain 32.4 (+80%) ← each doubling earns the same proportion

LLM training loss follows power-law decay: **when model size doubles, loss decreases by a fixed proportion.**

Written as a formula: `Loss(N) ≈ a × N^(-b) + c`

Where:
- N = number of model parameters
- a, b, c = constants (fitted from experiments)
- N^(-b) means "as N grows larger, this term shrinks"
- c represents the "ultimate floor of loss"—no matter how large the model, loss cannot go below c

In [ ]:
# === Hand-calculated demo of power-law decay ===
print("=== Power-law decay: Loss vs model size ===")
print()

# Simulate loss = a * N^(-b) + c
# Using small numbers for demonstration
a, b_pow, c = 5.0, 0.05, 2.0

# Models from 1M to 100B parameters
sizes = [1, 10, 100, 1000, 10000, 100000]  # unit: million parameters
labels = ["1M", "10M", "100M", "1B", "10B", "100B"]

print(f"Formula: Loss = {a} × N^(-{b_pow}) + {c}")
print()
print(f"{'Model size':>12s} {'Loss':>8s} {'vs previous':>14s}")
print("-" * 36)

prev_loss = None
for size, label in zip(sizes, labels):
    N = size * 1e6  # convert to parameter count
    loss = a * N**(-b_pow) + c

    if prev_loss:
        improvement = (prev_loss - loss) / prev_loss * 100
        print(f"{label:>12s}  {loss:>6.2f}  dropped {improvement:>5.1f}%")
    else:
        print(f"{label:>12s}  {loss:>6.2f}   —")
    prev_loss = loss

print()
print("Key observations:")
print("  1. Each time the model grows 10x, loss does decrease")
print("  2. But the decrease gets smaller each time → diminishing marginal returns")
print("  3. The drop from 1B → 10B is smaller than from 1M → 10M")
print()
print("This is the core conclusion of the power law: bigger is better, but less and less worthwhile.")
print("You need a formula to tell you: at what point is it no longer worth going bigger.")

## 2. Kaplan Scaling Laws (OpenAI, 2020) — "Prioritize Increasing Model Size"

In 2020, OpenAI ran extensive experiments to answer: given a fixed compute budget C (say 1e20 FLOPs), what is the best ratio between model size N and data volume D?

#### 2.1 Core Finding

They found that the optimal ratio looks like this:

```
N_opt ∝ C^0.73    ← model size grows fast with compute budget
D_opt ∝ C^0.27    ← data volume grows slowly with compute budget
```

The exponent 0.73 > 0.27 → **model grows much faster than data.**

In plain terms: if your compute budget doubles, you should spend most of the extra budget on a larger model, and only a small portion on more data.

#### 2.2 Let's feel this with concrete numbers

In [ ]:
# === Kaplan scaling law hand calculation ===
print("=== Kaplan Scaling Laws: given a compute budget, what is the optimal configuration? ===")
print()

# Reference point: assume when C=1e20 FLOPs, N=1B, D=20B tokens
N_ref = 1e9   # 1 billion parameters
D_ref = 20e9  # 20 billion tokens
C_ref = 1e20  # compute budget

print(f"Reference: C={C_ref/1e20:.0f}×10²⁰ FLOPs → N={N_ref/1e9:.0f}B, D={D_ref/1e9:.0f}B tokens")
print()

# Compute budget keeps doubling
budgets = [1e20, 2e20, 4e20, 8e20, 1e21, 1e22]
budget_labels = ["1×", "2×", "4×", "8×", "10×", "100×"]

print(f"{'Compute':>12s}  {'Model size (Kaplan)':>20s}  {'Data (Kaplan)':>18s}  {'D/N ratio':>10s}")
print("-" * 66)

for C, label in zip(budgets, budget_labels):
    ratio = C / C_ref
    N_opt = N_ref * (ratio ** 0.73)
    D_opt = D_ref * (ratio ** 0.27)
    dn_ratio = D_opt / N_opt

    if N_opt >= 1e9:
        n_str = f"{N_opt/1e9:.1f}B"
    else:
        n_str = f"{N_opt/1e6:.0f}M"

    d_str = f"{D_opt/1e9:.1f}B tokens"
    print(f"{label:>12s}  {n_str:>20s}  {d_str:>18s}  {dn_ratio:>8.1f}x")

print()
print("Kaplan's conclusion:")
print("  • Larger compute budget → much larger model (far exceeding data growth)")
print("  • D/N ratio keeps shrinking → large models can \"get by\" with less data")
print("  • E.g., a 10B model only needs 40B tokens (rather than \"the more the better\" as intuition suggests)")

## 3. Chinchilla Scaling Laws (DeepMind, 2022) — "Model and Data Are Equally Important"

#### 3.1 Why Was Kaplan Wrong?

DeepMind re-ran the experiments and found that when OpenAI increased compute budgets, **they did not sufficiently increase data**—because there was not that much high-quality data available at the time.

Kaplan's "prioritize model size" was actually a conclusion drawn under the premise of "not enough data."

#### 3.2 Chinchilla's New Conclusion

DeepMind controlled for variables: for each compute budget, they tried dozens of different (N, D) combinations and found the truly optimal one. The conclusion:

```
N_opt ∝ C^0.5
D_opt ∝ C^0.5
→ Model and data are equally important!
→ D_opt ≈ 20 × N_opt  (about 20 tokens per parameter)
```

#### 3.3 With the same compute budget, Kaplan and Chinchilla make different choices

In [ ]:
# === Kaplan vs Chinchilla showdown ===
print("=== Kaplan vs Chinchilla: same budget, different plans ===")
print()

# A specific budget
C = 1e23  # roughly GPT-3 training scale
ratio = C / C_ref

N_k = N_ref * (ratio ** 0.73)
D_k = D_ref * (ratio ** 0.27)

N_c = N_ref * (ratio ** 0.5)
D_c = D_ref * (ratio ** 0.5)

print(f"Compute budget: {C/1e23:.0f} × 10²³ FLOPs")
print()
print(f"{'':>20s}  {'Model size':>12s}  {'Data volume':>14s}  {'D/N ratio':>10s}")
print("-" * 60)
print(f"{'Kaplan says':>20s}  {N_k/1e9:>8.1f}B    {D_k/1e9:>9.1f}B tokens  {D_k/N_k:>8.1f}x")
print(f"{'Chinchilla says':>20s}  {N_c/1e9:>8.1f}B    {D_c/1e9:>9.1f}B tokens  {D_c/N_c:>8.1f}x")
print()

print(f"Comparison:")
print(f"  Kaplan plan:     model is {N_k/N_c:.1f}× larger, but data is only {D_k/D_c:.1f}× of Chinchilla")
print(f"  Chinchilla plan:  smaller model, but more data")
print()
print(f"Experimental result: Chinchilla plan achieves lower loss → overturns Kaplan")
print(f"          → large model with less data < medium model with more data")

#### 3.4 Chinchilla's Stunning Case Study

DeepMind used this finding to train two models:

```
Gopher:     280B params, trained on 300B tokens — Kaplan recipe
Chinchilla:  70B params, trained on 1.4T tokens  — Chinchilla recipe
          Same training compute!

Result: Chinchilla (70B) >> Gopher (280B)
      → A model 4× smaller, but trained on 4.7× more data, actually performs better
```

This was the shock of Chinchilla—**many large models at the time were probably \"underfed\" on data.**

## 4. Post-Chinchilla Era: Overtraining Is Actually Better

Chinchilla says D/N ≈ 20 is optimal. But actual practice in 2023-2024 completely broke past this recommendation.

#### 4.1 Why? Because Chinchilla Optimizes for "Training Cost"

Chinchilla's "optimal" means: **given a fixed training budget, achieve the lowest loss.**

But in actual deployment, **inference cost dominates**—after a model goes live serving millions of users, the money spent on inference far exceeds the training cost.

So the optimal strategy became: **train a small model, feed it tons of data, and let it exceed expectations.**

```
LLaMA 3 8B:  Chinchilla says it only needs 8B×20 = 160B tokens
            Actually trained on 15T tokens → 94× the Chinchilla optimal!
            Result → far exceeds what a Chinchilla-optimal 8B model should achieve
```

This is called the "inference-training tradeoff": spend more on training to save big on inference with a smaller model.

In [ ]:
# === D/N ratios of real models ===
print("=== D/N Ratios of Major Models ===")
print()

real_models = [
    ("Chinchilla (optimal)", 70, 1400, "20x"),
    ("LLaMA 7B", 7, 1000, "143x"),
    ("LLaMA 2 7B", 7, 2000, "286x"),
    ("LLaMA 3 8B", 8, 15000, "1875x"),
    ("DeepSeek-V2", 236, 8100, "34x (active: ~386x)"),
]

print(f"{'Model':<22s} {'Params':>8s} {'Data':>14s} {'D/N':>12s}")
print("-" * 58)
for name, params_b, tokens_b, dn in real_models:
    print(f"{name:<22s} {params_b:>5}B   {tokens_b:>6}B tokens  {dn:>12s}")

print()
print("Observation: post-2023 models have D/N ratios far exceeding 20x")
print("  → The industry consensus has become \"small model + massive data\"")
print("  → Because inference with a small model is much cheaper, spending more on training data is worth it")

## 5. µP: Maximal Update Parameterization

Before training a 70B model, you need to determine hyperparameters: learning rate, initialization standard deviation, weight decay coefficient... But grid search on a 70B model is unrealistic—a single run costs hundreds of thousands of dollars.

The common approach: tune parameters on a small 10M model, then transfer them as-is to 70B.

**But this method often fails.** The optimal learning rate on 10M may cause training instability on 70B—gradient explosion or loss plateau. The reason is that models of different sizes have different activation and gradient magnitudes during early training. A step size appropriate for a 10M model may be too large or too small for a 70B model.

µP solves exactly this problem. It applies special scaling rules to the initialization variance and learning rate of each layer, so that models of different sizes behave consistently during early training—a 10M model and a 70B model "feel" the same.

Specifically, µP controls the initialization variance and corresponding learning rate scaling for each layer's weight matrix. The core rule is: for a weight matrix W of hidden dimensions (d_out × d_in), initialization variance scales as 1/d_in, and learning rate scales as 1/d_in (for input weights) or 1/d_out (for output weights). This way, no matter how the model width changes, the output magnitude and gradient magnitude of each layer remain constant.

```
Without µP:  tune lr=0.01 on 10M → transfer to 70B → may become unstable, need to re-tune
With µP:     tune lr=0.01 on 10M → transfer to 70B → more likely to transfer, but still needs validation
```

This saves enormous amounts of human effort and compute: hyperparameters tuned on a small model are more likely to transfer to a large model, though real training still requires validation and minor adjustments. µP is an important method in large-model hyperparameter transfer research, and is adopted by many training systems and research projects. References: [µP paper](https://arxiv.org/abs/2203.03466), [microsoft/mup](https://github.com/microsoft/mup), [Cerebras µP docs](https://docs.cerebras.net/en/latest/wsc/Model-zoo/Model-Zoo-Components/Scaling-with-Maximal-Update-Parameterization.html).


## Second Half: Resource Estimation

## 6. FLOPs Estimation

#### 6.1 Core Formula

```
C ≈ 6 × P × D
```

Where:
- C = total compute (FLOPs)
- P = number of model parameters
- D = number of training tokens

#### 6.2 Why 6? Let's Calculate by Hand

Each training token requires forward + backward passes:

**Forward pass**:
- Mostly matrix multiplications.
- Multiplying an m×k matrix by a k×n matrix requires m×n×k multiply-adds = 2×m×n×k FLOPs
- Per token, total compute ≈ 2P FLOPs

**Backward pass**:
- Compute needed for gradients ≈ 2 × Forward = 4P FLOPs per token

**Total**: 2P + 4P = **6P** FLOPs per token.

Training D tokens → total FLOPs = 6PD.

#### 6.3 Let's Calculate with a Concrete Model

In [ ]:
# === FLOPs estimation: step by step ===
print("=== FLOPs Estimation: hand calculation ===")
print()

# Question: training LLaMA 7B on 1T tokens, how many FLOPs are needed?
P = 7e9     # 7B parameters
D = 1e12    # 1T tokens

print(f"Model params P = {P/1e9:.0f}B")
print(f"Training data D = {D/1e12:.0f}T tokens")
print()

# Step 1: FLOPs per token
flops_per_token = 6 * P
print(f"Step 1 — FLOPs per token = 6 × {P/1e9:.0f}B")
print(f"         = 6 × {P:.1e}")
print(f"         = {flops_per_token:.1e} FLOPs/token")
print()

# Step 2: Total FLOPs
total_flops = flops_per_token * D
print(f"Step 2 — Total FLOPs = {flops_per_token:.1e} × {D/1e12:.0f}T")
print(f"         = {flops_per_token:.1e} × {D:.1e}")
print(f"         = {total_flops:.1e} FLOPs")
print()

# Step 3: Convert to human-readable
def format_flops(flops):
    if flops >= 1e24:
        return f"{flops/1e24:.1f} YFLOPs (1e24)"
    elif flops >= 1e21:
        return f"{flops/1e21:.1f} ZFLOPs (1e21)"
    elif flops >= 1e18:
        return f"{flops/1e18:.1f} EFLOPs (1e18)"
    else:
        return f"{flops/1e15:.1f} PFLOPs (1e15)"

print(f"Step 3 — Result: {format_flops(total_flops)}")
print()
print("Next section converts this number into actual GPU time and cost.")

#### 6.4 From FLOPs to GPU-hours

FLOPs tell us "how many total operations are needed," but this number is so large (often 10²²) that it is hard to intuitively grasp "how many GPUs, for how long."

A more practical engineering metric is **GPU-hours**:

```
GPU-hours = Total FLOPs ÷ (effective compute per GPU × 3600)
```

"Effective compute per GPU" is not the same as the hardware's peak rating. In actual training, communication overhead, data loading, memory fragmentation, and other factors prevent GPUs from running at 100% utilization. Typical effective utilization is 40%-60%. Different GPUs have different peak ratings, utilization rates, and rental prices—choosing the right hardware has a big impact on cost. Let's calculate with concrete numbers.

In [ ]:
# === GPU comparison ===
print("=== Common Training GPUs Comparison (sample estimates) ===")
print()

# Prices change fast. The price here is not a quote commitment, just for demonstrating the calculation.
# Written on 2026-06-09. Real budgets must be recomputed using current cloud provider / rental platform quotes.
gpus = [
    ("A100 80GB", 312, 0.50, 2.0),
    ("H100 80GB", 990, 0.50, 3.5),
    ("H200 141GB", 990, 0.50, 4.5),
    ("RTX 4090", 165, 0.40, 0.5),
]

print(f"{'GPU':<18s} {'Peak':>8s} {'Util':>6s} {'Effective':>12s} {'Sample $/hr':>12s}")
print("-" * 58)
for name, peak, util, price in gpus:
    eff = peak * util
    print(f"{name:<18s} {peak:>5.0f} TF  {util*100:>4.0f}%  "
          f"{eff:>8.0f} TFLOPS   ${price:.1f}/h")

print()
print("Key observations:")
print("  • Peak compute only indicates the hardware ceiling; real speed also depends on utilization and communication overhead")
print("  • The sample hourly rate only demonstrates the calculation method; real prices vary by region, supply/demand, and rental period")
print("  • Don't look only at the per-GPU price; consider effective compute, memory capacity, network, and stability together")
print()

# --- Hand-calculate GPU-hours for LLaMA 7B + 1T tokens ---
P = 7e9
D = 1e12
total_flops = 6 * P * D

print("=== GPU-hours hand calculation: LLaMA 7B + 1T tokens ===")
print()
print(f"Total FLOPs = {total_flops:.1e}")
print()
print("Formula: GPU-hours = Total FLOPs ÷ (effective compute × 3600)")
print()

print(f"{'GPU':<18s} {'GPU-hours':>12s} {'1-GPU days':>12s} "
      f"{'256-GPU days':>12s} {'256-GPU cost':>14s}")
print("-" * 72)

for name, peak, util, price in gpus:
    eff = peak * util
    gpu_hours = total_flops / (eff * 1e12 * 3600)
    days_1 = gpu_hours / 24
    days_256 = gpu_hours / 256 / 24
    cost_256 = gpu_hours * price
    print(f"{name:<18s} {gpu_hours:>10,.0f} h  {days_1:>8,.0f} days  "
          f"{days_256:>8.1f} days  ${cost_256:>10,.0f}")

print()
print("Note: GPU-hours measure total compute, regardless of how many GPUs you use.")
print("      256 GPUs for 1 day ≈ 1 GPU for 256 days ≈ the same GPU-hours number.")
print("      More GPUs only shorten wall-clock time; real total cost is also affected by communication efficiency and queue time.")


In [ ]:
# === Major model training FLOPs and GPU-hours comparison ===
print("=== Mainstream Model Training Cost Overview (estimates) ===")
print()

models = [
    ("GPT-3 175B", 175e9, 300e9),
    ("LLaMA 7B", 7e9, 1e12),
    ("LLaMA 65B", 65e9, 1.4e12),
    ("Chinchilla 70B", 70e9, 1.4e12),
    ("LLaMA 2 70B", 70e9, 2e12),
    ("LLaMA 3 8B", 8e9, 15e12),
    ("LLaMA 3 70B", 70e9, 15e12),
    ("DeepSeek-V3", 671e9, 14.8e12),
]

# Use A100 as reference
a100_eff = 312 * 0.5  # A100 effective compute: 156 TFLOPS
a100_price = 2.0       # A100 reference rental: $2/h

print(f"{'Model':<18s} {'Params':>6s} {'Data':>9s} {'Total FLOPs':>12s} "
      f"{'GPU-hours':>12s} {'256-GPU days':>12s} {'Cost':>8s}")
print("-" * 80)

for name, params, tokens in models:
    flops = 6 * params * tokens
    gpu_hours = flops / (a100_eff * 1e12 * 3600)
    days_256 = gpu_hours / 256 / 24
    cost_m = gpu_hours * a100_price / 1e6

    print(f"{name:<18s} {params/1e9:>4.0f}B  {tokens/1e9:>6.0f}B tok "
          f"{format_flops(flops):>12s}  {gpu_hours:>10,.0f} h "
          f"{days_256:>7.1f} days ${cost_m:>5.1f}M")

print()
print("Calculation method:")
print("  GPU-hours = Total FLOPs ÷ (A100 effective compute 156 TFLOPS × 3600)")
print("  Cost = GPU-hours × $2/h (A100 reference price)")
print()
print("Notes:")
print("  • The above are lower-bound estimates for a single training run; actual costs include communication, retraining, etc.")
print("  • Different teams use different GPU types and utilization rates, so numbers may vary")

## 7. Memory Estimation

#### 7.1 What Eats Up GPU Memory?

During training, GPU memory houses four "tenants":

```
┌─────────────────────────────────────────┐
│          GPU Memory (VRAM)              │
├─────────────────────────────────────────┤
│ ① Model params (FP16)      2P bytes    │
│ ② Gradients (FP16)          2P bytes    │
│ ③ AdamW optimizer state (FP32)  8P bytes │
│    ├ m (momentum)           4P bytes    │
│    └ v (variance)           4P bytes    │
│ ④ Model param copy (FP32)  4P bytes    │
│ ⑤ Activations               ~2-10P      │
├─────────────────────────────────────────┤
│ Total                       ≈ 18-26P    │
└─────────────────────────────────────────┘
```

Rule of thumb: **each parameter requires about 20 bytes of memory.**

#### 7.2 Let's Calculate for LLaMA 7B

In [ ]:
# === Memory estimation: hand calculation ===
print("=== Memory Estimation: how much VRAM does LLaMA 7B full training need? ===")
print()

P = 7e9  # 7B parameters

# This is a rough starting point for AdamW + mixed precision + full training.
# Different frameworks will change memory due to ZeRO/FSDP, activation checkpointing, 8-bit optimizer, etc.
param_fp16 = 2 * P
grad_fp16 = 2 * P
adam_m = 4 * P
adam_v = 4 * P
param_fp32 = 4 * P

model_optimizer = param_fp16 + grad_fp16 + adam_m + adam_v + param_fp32

print("Model params + gradients + AdamW state:")
print(f"  ① Model params (FP16/BF16): {param_fp16/1e9:.1f} GB")
print(f"  ② Gradients (FP16/BF16):     {grad_fp16/1e9:.1f} GB")
print(f"  ③ Adam m (FP32):        {adam_m/1e9:.1f} GB")
print(f"  ④ Adam v (FP32):        {adam_v/1e9:.1f} GB")
print(f"  ⑤ FP32 master weights:  {param_fp32/1e9:.1f} GB")
print(f"  Subtotal:               {model_optimizer/1e9:.1f} GB")
print(f"  = {model_optimizer/P:.0f} bytes/param, excluding activations")
print()

# Activations are not a fixed bytes/param; they are mainly determined by batch_size, seq_len, num_layers, and checkpoint strategy.
batch_size = 4
seq_len = 4096

print(f"Activations (depend on batch={batch_size}, seq_len={seq_len} and whether recomputation is used):")
print(f"  Rough range: ~2-10P = {2*P/1e9:.0f}-{10*P/1e9:.0f} GB")
print()

activation_low = 2 * P
activation_high = 10 * P
total_low = (model_optimizer + activation_low) / 1e9
total_high = (model_optimizer + activation_high) / 1e9

print(f"Rough total VRAM: {total_low:.0f} - {total_high:.0f} GB")
print(f"A100 (80G) needed: {total_low/80:.0f} - {total_high/80:.0f} GPUs")
print()

print("=== Training Memory Requirements by Model (rough starting point) ===")
print()
print(f"{'Model':<18s} {'State+Grad':>12s} {'With act. (est.)':>16s} {'A100(80G)':>12s}")
print("-" * 62)

for name, params, _ in [
    ("1.5B small model", 1.5e9, None),
    ("LLaMA 7B", 7e9, None),
    ("LLaMA 13B", 13e9, None),
    ("LLaMA 65B", 65e9, None),
    ("GPT-3 175B", 175e9, None),
]:
    states = 16 * params / 1e9
    total = 20 * params / 1e9
    gpus = total / 80
    print(f"{name:<18s} {states:>8.0f} GB   ~{total:>10.0f} GB      {gpus:>8.0f} GPUs")

print()
print("Rule of thumb: 20 GB / 1B params is just a rough starting point for AdamW full training.")
print("ZeRO/FSDP partitions state, checkpointing saves activations, 8-bit optimizer saves optimizer state.")
print("Inference memory is mainly weights + KV Cache; the training memory formula cannot be applied directly.")


## 8. Summary of Three Scaling Law Paradigm Shifts

```
2020 Kaplan (OpenAI):
  "Prioritize increasing model size, data can be less"
  N ∝ C^0.73, D ∝ C^0.27
  → GPT-3 175B + 300B tokens (D/N=1.7x)

2022 Chinchilla (DeepMind):
  "Model and data are equally important!"
  N ∝ C^0.5, D ∝ C^0.5, D/N ≈ 20
  → Chinchilla 70B + 1.4T tokens (D/N=20x) beat Gopher 280B

2023+ Overtraining (LLaMA 3, etc.):
  "Small model + massive data, cheaper to infer"
  D/N >> 20, even > 1000
  → LLaMA 3 8B + 15T tokens (D/N=1875x)
```

**Each paradigm shift overturned a piece of "conventional wisdom"—this is how science and engineering intertwine to advance.**

## Summary

Confirm you understand these (check in order):

1. ✅ Power law: Loss ∝ N^(-b), diminishing returns—each doubling gives less improvement
2. ✅ Kaplan: prioritize model size, D grows slower than N
3. ✅ Chinchilla: model and data equally important, D/N ≈ 20
4. ✅ Overtraining: inference cost > training cost → small model with more data is more cost-effective
5. ✅ µP: enables hyperparameter sharing across model sizes, reducing tuning cost
6. ✅ FLOPs formula: C ≈ 6PD (forward 2P + backward 4P per token)
7. ✅ GPU-hours = Total FLOPs ÷ (effective compute × 3600); GPU-hours are independent of GPU count
8. ✅ Memory estimation: 20 bytes/param is a rough starting point for AdamW full training, not a fixed formula
9. ✅ Real memory also depends on ZeRO/FSDP, activation checkpointing, optimizer, batch, and seq_len

**One-sentence summary**: Scaling laws tell you the optimal ratio (theory), while FLOPs/memory/GPU-hours estimation tells you whether it is feasible and how much it costs (engineering). The 2024 consensus is—small model + massive data, optimizing for inference cost.

## Exercises> You can ask AI to help explain the approach, but it is not recommended to let AI "do this problem for you" directly.

**Exercise 1: FLOPs Estimation** Training LLaMA 7B used 1T ($10^{12}$) tokens. Use the formula $C \approx 6PD$ to compute the total FLOPs, and convert it into PFLOPs-days (1 PFLOPs-day = $10^{15} \times 86400$ FLOPs). Hint: $P = 7 \times 10^9$, $D = 10^{12}$, $C = 6 \times P \times D$.

In [ ]:
# Exercise 1: FLOPs Estimation
P = 7e9    # 7B parameters
D = 1e12   # 1T tokens
# TODO: compute total FLOPs
total_flops = None  # compute here
# TODO: convert to PFLOPs-days
# 1 PFLOPs-day = 1e15 * 86400 FLOPs
pflops_days = None  # compute here
assert total_flops is not None, "Please compute total FLOPs first"
assert pflops_days is not None, "Please compute PFLOPs-days first"
expected_flops = 6 * P * D
expected_days = expected_flops / (1e15 * 86400)
assert total_flops == expected_flops, f"Total FLOPs should be {expected_flops:.2e}"
assert abs(pflops_days - expected_days) < 0.01, f"PFLOPs-days should be {expected_days:.1f}"
print(f"✅ Exercise 1 passed:")
print(f"   Total FLOPs: {total_flops:.2e}")
print(f"   Equivalent to {pflops_days:.0f} PFLOPs-days")
print("   This roughly requires training on 2048 A100s for about 21 days.")

**Exercise 2: Kaplan vs Chinchilla Resource Allocation** Given a compute budget $C$, Kaplan suggests allocating 73% of the budget growth to model parameters ($N \propto C^{0.73}$), while Chinchilla suggests 50% each for model and data ($N \propto C^{0.5}$). If the budget doubles ($C$ becomes $2C$), compute how many times the model parameters $N$ grow under the Kaplan and Chinchilla plans respectively. Hint: $N_{new}/N_{old} = 2^{\text{exponent}}$, use 0.73 for Kaplan and 0.5 for Chinchilla.

In [ ]:
# Exercise 2: Kaplan vs Chinchilla Resource Allocation
import math
# Budget doubles: ratio = 2
ratio = 2
# TODO: compute model parameter growth factor under Kaplan
# N_new / N_old = ratio ^ 0.73
kaplan_ratio = None  # compute here
# TODO: compute model parameter growth factor under Chinchilla
# N_new / N_old = ratio ^ 0.5
chinchilla_ratio = None  # compute here
assert kaplan_ratio is not None, "Please compute Kaplan ratio first"
assert chinchilla_ratio is not None, "Please compute Chinchilla ratio first"
expected_kaplan = 2 ** 0.73
expected_chinchilla = 2 ** 0.5
assert abs(kaplan_ratio - expected_kaplan) < 0.01, f"Kaplan ratio should be {expected_kaplan:.3f}"
assert abs(chinchilla_ratio - expected_chinchilla) < 0.01, f"Chinchilla ratio should be {expected_chinchilla:.3f}"
print(f"✅ Exercise 2 passed:")
print(f"   Kaplan: model params grow {kaplan_ratio:.2f}× (prioritize model size)")
print(f"   Chinchilla: model params grow {chinchilla_ratio:.2f}× (balanced model/data growth)")
print("   Kaplan's model grows faster, Chinchilla is more balanced.")

**Exercise 3: Memory Estimation** LLaMA 7B full training (AdamW + mixed precision) memory composition:
- Model params (FP16): $2P$
- Gradients (FP16): $2P$
- AdamW optimizer state (FP32 m and v): $8P$
- Model param master copy (FP32): $4P$
Compute the total memory requirement (in GB, $P = 7 \times 10^9$), and explain why a 7B model in full training needs far more than 14 GB of memory. Hint: total memory = $(2 + 2 + 8 + 4) \times P$ bytes = $16P$ bytes, then convert to GB.

In [ ]:
# Exercise 3: Memory Estimation
P = 7e9  # 7B parameters
# TODO: compute each part of memory (in bytes)
param_fp16 = None     # 2 bytes per param
grad_fp16 = None      # 2 bytes per param
adam_m = None          # 4 bytes per param (FP32)
adam_v = None          # 4 bytes per param (FP32)
param_fp32 = None      # 4 bytes per param (master copy)
# TODO: compute total memory (in GB)
total_memory_gb = None  # compute here
assert all(v is not None for v in [param_fp16, grad_fp16, adam_m, adam_v, param_fp32]), "Please compute each part first"
assert total_memory_gb is not None, "Please compute total memory first"
expected = (2*P + 2*P + 4*P + 4*P + 4*P) / 1e9
assert abs(total_memory_gb - expected) < 0.1, f"Total memory should be {expected:.1f} GB"
print(f"✅ Exercise 3 passed:")
print(f"   Params FP16: {param_fp16/1e9:.1f} GB")
print(f"   Gradients FP16: {grad_fp16/1e9:.1f} GB")
print(f"   Adam m:    {adam_m/1e9:.1f} GB")
print(f"   Adam v:    {adam_v/1e9:.1f} GB")
print(f"   Params FP32: {param_fp32/1e9:.1f} GB")
print(f"   Total: {total_memory_gb:.1f} GB (excluding activations)")
print("   7B model full training needs ~112 GB, far more than the 14 GB of the params alone.")